In [4]:
#import packages
import schwabdev
import logging
import dotenv
import os
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd().resolve().parent.parent.parent

# set logging level
logging.basicConfig(level=logging.INFO)

#load environment variables and make client
dotenv.load_dotenv(BASE_DIR/ '.env')

PosixPath('/home/zhaohuiwang/dev/finance-projects')

In [ ]:
client = schwabdev.Client(os.getenv('APP_KEY'), os.getenv('APP_SECRET'), os.getenv('CALLBACK_URL'))
# del client # To clean up

INFO:Schwabdev:Access token expires in: 0:27:52
INFO:Schwabdev:Refresh token expires in: 6 days, 23:57:52


# Basic API calls

In [ ]:
# get account number and hashes for linked accounts
linked_accounts = client.linked_accounts().json()
print(linked_accounts)
# select the first account to use for orders
account_hash = linked_accounts[0].get('hashValue')

In [27]:
# get positions for selected account
print(client.account_details(account_hash, fields="positions").json())

{'securitiesAccount': {'type': 'MARGIN', 'accountNumber': '29308909', 'roundTrips': 0, 'isDayTrader': True, 'isClosingOnlyRestricted': False, 'pfcbFlag': False, 'positions': [{'shortQuantity': 0.0, 'averagePrice': 8.826, 'currentDayProfitLoss': -551.000000000004, 'currentDayProfitLossPercentage': -2.63, 'longQuantity': 2900.0, 'settledLongQuantity': 2900.0, 'settledShortQuantity': 0.0, 'instrument': {'assetType': 'EQUITY', 'cusip': '03945R102', 'symbol': 'ACHR', 'netChange': -0.19}, 'marketValue': 20387.0, 'maintenanceRequirement': 10193.5, 'averageLongPrice': 8.826, 'taxLotAverageLongPrice': 8.826, 'longOpenProfitLoss': -5208.400000000001, 'previousSessionLongQuantity': 2900.0, 'currentDayCost': 0.0}, {'shortQuantity': 0.0, 'averagePrice': 113.98, 'currentDayProfitLoss': -3.77, 'currentDayProfitLossPercentage': -4.08, 'longQuantity': 1.0, 'settledLongQuantity': 1.0, 'settledShortQuantity': 0.0, 'instrument': {'assetType': 'EQUITY', 'cusip': 'N97284108', 'symbol': 'NBIS', 'netChange': 

In [28]:
# get a list of quotes
print(client.quotes(["AAPL", "AMD"]).json())

{'AAPL': {'assetMainType': 'EQUITY', 'assetSubType': 'COE', 'quoteType': 'NBBO', 'realtime': True, 'ssid': 1973757747, 'symbol': 'AAPL', 'extended': {'askPrice': 276.4, 'askSize': 100, 'bidPrice': 275.8, 'bidSize': 209, 'lastPrice': 276.25, 'lastSize': 1, 'mark': 276.25, 'quoteTime': 1770868038000, 'totalVolume': 0, 'tradeTime': 1770868032000}, 'fundamental': {'avg10DaysVolume': 62754070.0, 'avg1YearVolume': 53923963.0, 'declarationDate': '2026-01-29T00:00:00Z', 'divAmount': 1.04, 'divExDate': '2026-02-09T00:00:00Z', 'divFreq': 4, 'divPayAmount': 0.26, 'divPayDate': '2026-02-12T00:00:00Z', 'divYield': 0.38001, 'eps': 7.46, 'fundLeverageFactor': 0.0, 'lastEarningsDate': '2026-01-29T00:00:00Z', 'nextDivExDate': '2026-05-11T00:00:00Z', 'nextDivPayDate': '2026-05-12T00:00:00Z', 'peRatio': 34.71083, 'sharesOutstanding': 14681140000}, 'quote': {'52WeekHigh': 288.62, '52WeekLow': 169.2101, 'askMICId': 'XNAS', 'askPrice': 276.08, 'askSize': 600, 'askTime': 1770857943949, 'bidMICId': 'ARCX', 'b

In [29]:
# get an option chain
print(client.option_expiration_chain("AAPL").json())

{'expirationList': [{'expirationDate': '2026-02-11', 'daysToExpiration': 0, 'expirationType': 'W', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-13', 'daysToExpiration': 2, 'expirationType': 'W', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-18', 'daysToExpiration': 7, 'expirationType': 'W', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-20', 'daysToExpiration': 9, 'expirationType': 'S', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-23', 'daysToExpiration': 12, 'expirationType': 'W', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-25', 'daysToExpiration': 14, 'expirationType': 'W', 'settlementType': 'P', 'optionRoots': 'AAPL', 'standard': True}, {'expirationDate': '2026-02-27', 'daysToExpiration': 16, 'expirationType': 'W', 'settlementType': 'P', 'optionRoot

# Order example
This **WILL** place an on your account.

In [ ]:
# place an order for INTC at limit price $10.00
order = {"orderType": "LIMIT", 
         "session": "NORMAL", 
         "duration": "DAY", 
         "orderStrategyType": "SINGLE", 
         "price": '10.00',
         "orderLegCollection": [
             {"instruction": "BUY", 
              "quantity": 1, 
              "instrument": 
                  {"symbol": "INTC", 
                   "assetType": "EQUITY"
                   }
              }
         ]}
resp = client.place_order(account_hash, order)
print(f"Response code: {resp}") 

# get the order ID - if order is immediately filled then the id might not be returned
order_id = resp.headers.get('location', '/').split('/')[-1] 
print(f"Order id: {order_id}")

In [ ]:
# cancel the order
print(client.cancel_order(account_hash, order_id))

# Streaming example

In [ ]:
# create streamer
streamer = schwabdev.Stream(client)

In [ ]:
# create a list to store responses
responses = []
def add_to_list(message):
    responses.append(message)

In [ ]:
#start stream and send request
streamer.start(add_to_list)
streamer.send(streamer.level_one_equities("AMD", "0,1,2,3,4,5,6,7,8"))

In [ ]:
#check responses REMEMBER: the stream is running in the background so the responses list will change on subsequent reruns)
print(responses)

In [ ]:
#stop stream
streamer.stop()